# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 31: Diversidade entre Entradas — Par a Par e as Quatro ao Mesmo Tempo

**Autor:** Evandro Rocha (UFRJ / COPPE / PEE)
**Data:** Outubro de 2026

---

O Notebook 30 mediu, par a par, se MEL, LOFAR, HHT e DEMON erram juntas para o mesmo classificador (Q e Y de Yule); os modelos são os mesmos (na CNN, MEL e LOFAR são as CNNs do artigo, `test_scripts/cnn_mel_artigo.py` e `test_scripts/cnn_lofar.py`). Aqui:

1. **Par a par, com quatro medidas:** Q e Y de Yule (sobre os erros) e V de Cramér e κ de Cohen (sobre as classes preditas). Elas ordenam os pares do mesmo jeito?
2. **As quatro entradas ao mesmo tempo:** κ de Fleiss e a distribuição de quantas entradas erram cada gravação, contra o esperado se os erros fossem independentes.
3. **Qual entrada traz mais diversidade:** κ ao retirar uma entrada por vez.

**Resumo:**
- Com as quatro entradas, o κ de Fleiss sobre acerto/erro fica entre 0,33 e 0,41 nas 5 famílias completas: os erros estão longe de independentes (κ = 0), mas também longe de idênticos (κ = 1).
- As quatro entradas erram juntas a mesma gravação 4,7–6,3× mais do que se fossem independentes; o teste χ² rejeita a independência em todos os folds (p < 0,001 em todos).
- A entrada que mais traz diversidade é **DEMON**: retirá-la aumenta o κ de Fleiss em média 0,083 (de 0,045 a 0,116 entre as famílias completas).
- As medidas par a par concordam na ordem: MEL × LOFAR é o par mais associado e os pares com DEMON os menos associados.

Predição por gravação = voto majoritário das janelas; todas as tabelas são calculadas na execução (`notebooks/src/iara_results.py`).

In [ ]:
import os, sys, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, chisquare
from sklearn.metrics import cohen_kappa_score
try:
    from IPython.display import display
except ImportError:
    display = print

for _p in ('.', 'src', 'notebooks', 'notebooks/src'):
    if os.path.exists(os.path.join(_p, 'iara_results.py')) and os.path.abspath(_p) not in sys.path:
        sys.path.insert(0, os.path.abspath(_p))
import iara_results as R

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({'font.family': 'sans-serif', 'font.size': 10, 'figure.dpi': 120})
pd.set_option('display.precision', 2)
REPS = ['MEL', 'LOFAR', 'HHT', 'DEMON']


def disponivel(exp):
    try:
        R.file_predictions(exp)
        return True
    except FileNotFoundError:
        return False


# Mesmos modelos do Notebook 30: um por (família, entrada), 4 classes, 10 folds oficiais (split por ID)
FAM = {
    'RF': {'MEL': 'rf_mel5s_n100_balanced_4c', 'LOFAR': 'rf_lofar5s_n100_balanced_4c',
           'HHT': 'rf_hht_n100_balanced_4c', 'DEMON': 'rf_demon_n100_balanced_4c'},
    'SVM': {'MEL': 'svm_nystroem_4000_log_melgram_elasticnet_l1r0.15_C2.0',
            'LOFAR': 'svm_nystroem_6000_lofar_elasticnet_l1r0.15',
            'HHT': 'svm_nystroem_6000_hht_C2.0_elasticnet_l1r0.15_4c',
            'DEMON': 'svm_nystroem_6000_demon_C2.0_elasticnet_l1r0.15_it1000_4c'},
    'MLP': {'MEL': 'mlp', 'LOFAR': 'mlp_lofar', 'HHT': 'mlp_hht_h32-16_e30_4c', 'DEMON': 'mlp_demon_h32-16_e30_4c'},
    'BiGRU': {'MEL': 'bigru_att_mel5s_h64_s15-5_e20_4c', 'LOFAR': 'bigru_att_lofar5s_h128_s15-5_e15_4c',
              'HHT': 'bigru_att_hht_h64_s15-5_e20_4c', 'DEMON': 'bigru_att_demon_h64_s15-5_e20_4c'},
    'CNN': {'MEL': 'cnn_mel_artigo', 'LOFAR': 'cnn_lofar', 'HHT': 'hht_2d_cnn_e40p7_4c', 'DEMON': 'hcnn_demon_tpsw_med5_e60p8_4c'},
}
for f in FAM:
    FAM[f] = {r: e for r, e in FAM[f].items() if disponivel(e)}
COMPLETAS = [f for f in FAM if len(FAM[f]) == 4]
print({f: list(r) for f, r in FAM.items()})
print('Famílias com as 4 entradas:', COMPLETAS)

## 1. As medidas

**Par a par.**

| Medida | Sobre | Tabela | Faixa | O que mede |
|---|---|---|---|---|
| Q de Yule | erros | 2×2 (acerta/erra × acerta/erra) | −1 a +1 | associação dos erros; Q = (OR − 1)/(OR + 1) |
| Y de Yule (colligação) | erros | 2×2 | −1 a +1 | idem, comprimida; Y = (√OR − 1)/(√OR + 1) |
| V de Cramér | classes preditas | 4×4 (classe de A × classe de B) | 0 a 1 | qualquer associação entre as predições: V = √(χ² / (N·(k − 1))) |
| κ de Cohen | classes preditas | 4×4 | −1 a +1 | concordância além do acaso (só a diagonal conta) |

O V de Cramér também é uma medida de **duas** variáveis; a diferença para o Q é aceitar mais de duas categorias. Ele não mede concordância: se A dissesse sempre `LARGE` quando B diz `SMALL`, V seria 1 e κ seria negativo. Por isso vem acompanhado do κ de Cohen.

**Quanto precisa dar para indicar concordância?** Não há ponto de corte formal, só escalas de convenção:

| κ de Cohen (Landis e Koch, 1977) | Leitura | | V de Cramér (Rea e Parker) | Leitura |
|---|---|---|---|---|
| < 0 | pior que o acaso | | < 0,10 | desprezível |
| 0,00–0,20 | leve | | 0,10–0,20 | fraca |
| 0,21–0,40 | razoável | | 0,20–0,40 | moderada |
| 0,41–0,60 | moderada | | 0,40–0,60 | relativamente forte |
| 0,61–0,80 | substancial | | 0,60–0,80 | forte |
| 0,81–1,00 | quase perfeita | | > 0,80 | muito forte |

McHugh (2012) é mais exigente (κ < 0,60 já é concordância fraca). Para o V, Cohen (1988) usa limiares que dependem dos graus de liberdade: com 4 classes (df* = 3), 0,06 / 0,17 / 0,29 = efeito pequeno / médio / grande. Com centenas de gravações por fold, qualquer κ ou V acima de ~0,1 é significativo (p < 0,001); o que importa é o tamanho.

**A referência não é κ = 0.** Dois classificadores que acertam 60–70% concordam muito só por acertarem as mesmas classes, mesmo que errem de forma independente. A referência justa para "erram juntos?" é o **κ esperado se os erros fossem independentes dada a classe verdadeira**: cada entrada mantém a sua matriz de confusão observada, mas as duas erram independentemente, P(A = i, B = j) = Σ_c π_c · P_A(i | c) · P_B(j | c). A coluna **κ − κ indep.** é a concordância que sobra além do que as duas acurácias já explicam.

**As quatro de uma vez** (medidas não pareadas, Kuncheva e Whitaker, 2003):
- **κ de Fleiss sobre acerto/erro:** trata as entradas como avaliadores que dão a cada gravação o rótulo "acertou" ou "errou". κ = 0: erros independentes; κ = 1: todas erram e acertam sempre juntas; κ < 0: mais diversas que o acaso. Para um ensemble, **quanto menor, melhor**. É a medida de concordância entre avaliadores de Kuncheva e Whitaker.
- **κ de Fleiss sobre as classes preditas:** a concordância entre as quatro nas 4 classes.
- **Número de entradas que erram cada gravação:** se os erros fossem independentes, esse número seguiria uma binomial de Poisson com as taxas de erro de cada entrada. O excesso nas pontas (todas erram, todas acertam) mostra a dependência; o teste χ² de aderência compara as duas distribuições em cada fold.

Todas as medidas são calculadas em cada fold (639 gravações de teste) e depois tirada a média dos 10 folds.

In [ ]:
def yule(ea, eb):
    """Q e Y de Yule dos erros (a = ambos erram, d = ambos acertam, b/c = só um erra)."""
    n11, n00 = float((ea & eb).sum()), float((~ea & ~eb).sum())
    n10, n01 = float((ea & ~eb).sum()), float((~ea & eb).sum())
    ad, bc = n11 * n00, n10 * n01
    return (ad - bc) / (ad + bc), (np.sqrt(ad) - np.sqrt(bc)) / (np.sqrt(ad) + np.sqrt(bc))


def cramer_v(pa, pb):
    t = pd.crosstab(pa, pb).values
    chi2 = chi2_contingency(t, correction=False)[0]
    return np.sqrt(chi2 / (t.sum() * (min(t.shape) - 1)))


def fleiss(M, n_cat):
    """κ de Fleiss: M = (gravações × avaliadores) com rótulos 0..n_cat-1."""
    N, L = M.shape
    n = np.stack([(M == c).sum(1) for c in range(n_cat)], 1)
    pj = n.sum(0) / (N * L)
    Pi = (n * (n - 1)).sum(1) / (L * (L - 1))
    Pe = np.sum(pj ** 2)
    return (Pi.mean() - Pe) / (1 - Pe)


def kappa_indep(y, pa, pb, n_cat=4):
    """κ de Cohen esperado se A e B errassem de forma independente dada a classe verdadeira:
    P(A = i, B = j) = Σ_c π_c · P_A(i | c) · P_B(j | c), com as matrizes de confusão observadas."""
    J = np.zeros((n_cat, n_cat))
    for c in np.unique(y):
        m = y == c
        J += m.mean() * np.outer(np.bincount(pa[m], minlength=n_cat) / m.sum(),
                                 np.bincount(pb[m], minlength=n_cat) / m.sum())
    po, pe = np.trace(J), J.sum(1) @ J.sum(0)
    return (po - pe) / (1 - pe)


def poisson_binomial(ps):
    d = np.array([1.0])
    for p in ps:
        d = np.convolve(d, [1 - p, p])
    return d


AL = {f: R.aligned(list(reps.values())) for f, reps in FAM.items()}

## 2. Par a par: quatro medidas

In [ ]:
rows = []
for f, reps in FAM.items():
    for ra, rb in itertools.combinations([r for r in REPS if r in reps], 2):
        a, b = reps[ra], reps[rb]
        per = []
        for d in AL[f].values():
            y = d.Target.values
            q, yy = yule(d[a].values != y, d[b].values != y)
            k = cohen_kappa_score(d[a].values, d[b].values)
            ki = kappa_indep(y.astype(int), d[a].values.astype(int), d[b].values.astype(int))
            per.append({'Q (erros)': q, 'Y (erros)': yy, 'V de Cramér (classes)': cramer_v(d[a].values, d[b].values),
                        'κ de Cohen (classes)': k, 'κ se erros indep.': ki, 'κ − κ indep.': k - ki,
                        'concordância (%)': 100 * np.mean(d[a].values == d[b].values)})
        rows.append({'Família': f, 'Par': f'{ra} × {rb}', **pd.DataFrame(per).mean()})
PARES = pd.DataFrame(rows)
display(PARES.set_index(['Família', 'Par']).round(2))

ORDEM = [f'{a} × {b}' for a, b in itertools.combinations(REPS, 2)]
MED = PARES.groupby('Par')[['Q (erros)', 'Y (erros)', 'V de Cramér (classes)', 'κ de Cohen (classes)',
                            'κ se erros indep.', 'κ − κ indep.']].mean()
MED = MED.reindex([p for p in ORDEM if p in MED.index]).sort_values('Q (erros)', ascending=False)
print('Média das famílias, por par de entradas:')
display(MED.round(2))

fig, ax = plt.subplots(figsize=(7, 4.5))
for col, mk in (('Q (erros)', 'o'), ('Y (erros)', 's'), ('V de Cramér (classes)', '^'), ('κ de Cohen (classes)', 'D')):
    ax.plot(range(len(MED)), MED[col], marker=mk, label=col)
ax.plot(range(len(MED)), MED['κ se erros indep.'], ls='--', color='gray', marker='D', mfc='none',
        label='κ de Cohen se erros indep.')
ax.set_xticks(range(len(MED)), MED.index, rotation=25, ha='right'); ax.set_ylim(0, 1)
ax.set_ylabel('valor (média das famílias)'); ax.legend(); ax.set_title('Pares de entradas: quatro medidas, mesma ordem')
plt.tight_layout(); plt.show()

**Leitura.**
- **Ordem pelo Q:** MEL × LOFAR, LOFAR × HHT, MEL × HHT, LOFAR × DEMON, HHT × DEMON, MEL × DEMON. **Pelo V de Cramér:** MEL × LOFAR, LOFAR × HHT, MEL × HHT, LOFAR × DEMON, MEL × DEMON, HHT × DEMON. **Pelo κ de Cohen:** MEL × LOFAR, LOFAR × HHT, MEL × HHT, LOFAR × DEMON, MEL × DEMON, HHT × DEMON.
- As ordens são parecidas, mas não idênticas: os pares com DEMON ficam no fim em todas.
- **κ observado × κ esperado com erros independentes:** MEL × LOFAR 0,60 vs 0,29; LOFAR × HHT 0,50 vs 0,25; MEL × HHT 0,49 vs 0,25; LOFAR × DEMON 0,34 vs 0,22; HHT × DEMON 0,32 vs 0,19; MEL × DEMON 0,33 vs 0,21. Parte do κ vem só de as entradas acertarem bastante; o excesso (κ − κ indep., de 0,12 a 0,31) é a concordância produzida pelos erros compartilhados; os três pares com DEMON têm o menor excesso.
- O V de Cramér vai de 0,34 a 0,71. V e κ de Cohen ficam próximos porque as associações entre as predições estão quase todas na diagonal (as entradas concordam na classe, não trocam uma classe por outra de forma sistemática).

## 3. As quatro entradas ao mesmo tempo

In [ ]:
def multi(f, reps_sel):
    ex = [FAM[f][r] for r in reps_sel]
    L = len(ex)
    ks, kc, obs, exp, pchi = [], [], np.zeros(L + 1), np.zeros(L + 1), []
    for d in AL[f].values():
        y = d.Target.values
        P = np.stack([d[e].values for e in ex], 1)
        C = (P == y[:, None]).astype(int)
        ks.append(fleiss(C, 2))
        kc.append(fleiss(P, 4))
        o = np.bincount(L - C.sum(1), minlength=L + 1)
        e = poisson_binomial(1 - C.mean(0)) * len(y)
        obs += o; exp += e
        pchi.append(chisquare(o, e)[1])
    return {'κ Fleiss acerto/erro': np.mean(ks), 'κ desvio (folds)': np.std(ks),
            'κ Fleiss classes': np.mean(kc), 'obs': obs / 10, 'exp': exp / 10, 'p χ² máx (folds)': max(pchi)}


MULTI, DIST = [], []
for f in FAM:
    reps_f = [r for r in REPS if r in FAM[f]]
    m = multi(f, reps_f)
    L = len(reps_f)
    MULTI.append({'Família': f, 'Entradas': ' + '.join(reps_f), 'κ Fleiss acerto/erro': m['κ Fleiss acerto/erro'],
                  'κ desvio (folds)': m['κ desvio (folds)'], 'κ Fleiss classes': m['κ Fleiss classes'],
                  'todas erram: obs.': m['obs'][L], 'todas erram: indep.': m['exp'][L],
                  'todas erram: obs./indep.': m['obs'][L] / m['exp'][L],
                  'todas acertam: obs.': m['obs'][0], 'todas acertam: indep.': m['exp'][0],
                  'p χ² (pior fold)': m['p χ² máx (folds)']})
    for k in range(L + 1):
        DIST.append({'Família': f, 'nº de entradas que erram': k, 'observado': m['obs'][k], 'independência': m['exp'][k]})
MULTI = pd.DataFrame(MULTI).set_index('Família')
DIST = pd.DataFrame(DIST)
display(MULTI.round(3))

fams = list(FAM)
fig, axes = plt.subplots(1, len(fams), figsize=(3.2 * len(fams), 3.4), sharey=True)
for ax, f in zip(axes, fams):
    d = DIST[DIST['Família'] == f]
    x = d['nº de entradas que erram'].values
    ax.bar(x - 0.2, d.observado, 0.4, label='observado', color='#C44E52')
    ax.bar(x + 0.2, d['independência'], 0.4, label='se independentes', color='#4C72B0')
    ax.set_xticks(x); ax.set_xlabel('nº de entradas que erram'); ax.set_title(f'{f} ({len(x) - 1} entradas)')
axes[0].set_ylabel('gravações por fold (média)'); axes[0].legend(fontsize=8)
plt.suptitle('Quantas entradas erram cada gravação: observado × esperado se os erros fossem independentes', y=1.02)
plt.tight_layout(); plt.show()

**Leitura.**
- **κ de Fleiss (acerto/erro) de 0,33 a 0,41** nas famílias com as quatro entradas: dependência clara, longe de independência e de redundância total. Sobre as classes preditas, κ de 0,38 a 0,47.
- **Todas erram juntas:** RF 92 gravações por fold, contra 18 esperadas; SVM 89 gravações por fold, contra 19 esperadas; MLP 93 gravações por fold, contra 16 esperadas; BiGRU 80 gravações por fold, contra 17 esperadas; CNN 95 gravações por fold, contra 15 esperadas. É o núcleo de gravações difíceis do Notebook 29 (navio fora do treino, comprimento perto dos limiares de 50 m e 100 m).
- **Todas acertam juntas** também acima do esperado, e os casos intermediários (1–2 entradas erram) abaixo: a distribuição observada é mais "polarizada" que a independente. As gravações tendem a ser fáceis para todas ou difíceis para todas.
- A CNN tem as quatro entradas.

## 4. Qual entrada traz mais diversidade?

Retiramos uma entrada por vez e recalculamos o κ de Fleiss (acerto/erro) com as três restantes. Se o κ **sobe** sem a entrada, ela era a que mais discordava das outras.

In [ ]:
# Retirar uma entrada por vez (só famílias com as 4): κ maior sem a entrada = ela trazia diversidade
loo = []
for f in COMPLETAS:
    k4 = MULTI.loc[f, 'κ Fleiss acerto/erro']
    linha = {'Família': f, 'κ com as 4': k4}
    for r in REPS:
        linha[f'sem {r}'] = multi(f, [x for x in REPS if x != r])['κ Fleiss acerto/erro'] - k4
    loo.append(linha)
LOO = pd.DataFrame(loo).set_index('Família')
print('Δκ de Fleiss (acerto/erro) ao retirar cada entrada; positivo = a entrada retirada trazia diversidade')
display(LOO.round(3))

**Leitura.**
- **Média do Δκ por entrada retirada:** MEL -0,031; LOFAR -0,043; HHT -0,010; DEMON 0,083.
- **O DEMON é a entrada que mais traz diversidade** em todas as famílias completas: sem ele, as três restantes (MEL, LOFAR, HHT) concordam bem mais entre si.
- Diversidade não é ganho: o DEMON é também a entrada mais fraca sozinha (Notebook 30, Seções 5 e 6). Um combinador só aproveita essa diversidade se souber quando confiar nele.

## 5. Conclusões

1. **As quatro medidas par a par contam a mesma história:** MEL × LOFAR é o par mais redundante e os pares com DEMON os mais complementares. O V de Cramér é uma medida de duas variáveis, como o Q; para comparar as quatro entradas de uma vez, a medida adequada é o κ de Fleiss (ou a distribuição do número de entradas que erram).
2. **Com as quatro entradas ao mesmo tempo, os erros não são independentes:** existe um núcleo de gravações que todas erram, várias vezes maior do que o acaso produziria.
3. **O DEMON é quem traz diversidade ao conjunto;** MEL, LOFAR e HHT são parecidas entre si. Melhorar o DEMON sozinho é o caminho para que essa diversidade vire ganho num combinador.

**Limites:**
- No MLP, MEL/LOFAR (0,512 s) e HHT/DEMON (5 s) vêm de pipelines diferentes.
- A CNN tem as quatro entradas.
- κ de Fleiss sobre acerto/erro depende das taxas de erro (como todo κ); comparar famílias com acurácias muito diferentes exige cuidado.
- O teste χ² de aderência é feito fold a fold; as gravações se repetem entre as 5 repetições, então os folds não são independentes entre si.